# Method 6 (short version) — Two-stage: flag outliers, then refill them from their neighbours

Same code and results as `method_6.ipynb`, without the vocabulary, worked examples and long explanations
(read the full version for those).

Notebook map:
1. Load data, shared code from the group doc, evaluation helpers.
2. Assumptions and formulas; method code; the two functions for the group (`flag_outliers`, `refill_outliers`).
3. Final `x_hat` on the real data, `results/method_6/xhat_6.npy`, figure in `figures/method_6/`.
4. Result row for the group comparison table.
5. Write-up skeleton (Mục VI).

Appendices: A. parameter check with `cv_score`; B. synthetic data; C. Stage 1 variants on all 100 public graphs.

## 1. Data and shared code

In [ ]:
# ---- Mục III/IV of the assignment: load G, y, positions ---------------------
# NOTE: replace this placeholder with the REAL representative MSSV of your group
# (same one used in the group notebook), otherwise you get a different graph.
import json, hashlib
from pathlib import Path
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Repository root = the folder that holds public_cases.json (works from any subfolder).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "public_cases.json").exists())
FIG_DIR = ROOT / "figures" / "method_6"            # figures of this method
RES_DIR = ROOT / "results" / "method_6"            # xhat_6.npy
FIG_DIR.mkdir(parents=True, exist_ok=True); RES_DIR.mkdir(parents=True, exist_ok=True)

MSSV_DAI_DIEN = "20260001"          # <-- change to the group's real representative MSSV

def assign_mssv_to_case(student_id, cases):
    # Same deterministic MSSV -> case mapping as in the assignment notebook.
    sid = str(student_id).strip()
    if sid in cases:
        return sid
    case_ids = sorted(cases, key=lambda x: (0, int(x)) if x.isdecimal() else (1, x))
    k = len(case_ids)
    if sid.isdecimal():
        index = int(sid) % k
    else:
        index = int.from_bytes(hashlib.sha256(sid.encode("utf-8")).digest(), "big") % k
    return case_ids[index]

def get_problem(student_id, file_path=ROOT / "public_cases.json"):
    db = json.loads(Path(file_path).read_text(encoding="utf-8"))
    case = db["cases"][assign_mssv_to_case(student_id, db["cases"])]
    G = nx.Graph()
    for i, xy in enumerate(case["positions"]):
        G.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in case["edges"]:
        G.add_edge(int(u), int(v), weight=float(w))
    y = np.asarray(case["y"], dtype=float)
    assert nx.is_connected(G) and len(y) == G.number_of_nodes()
    return G, y

G, y = get_problem(MSSV_DAI_DIEN)
positions = nx.get_node_attributes(G, "pos")
n = len(y)
print("n nodes =", n, "| n edges =", G.number_of_edges())

In [ ]:
# ---- Shared "contract" block copied from the group doc ----------------------
# L = weighted graph Laplacian = D - W.  For any signal x:  x^T L x = sum_{(i,j)} w_ij (x_i - x_j)^2
# i.e. x^T L x is SMALL when neighbouring nodes have similar values (a "smooth" signal).
n = len(y)
L = nx.laplacian_matrix(G, nodelist=range(n), weight="weight").toarray()

# (a) Synthetic data with a KNOWN x, so we can measure RMSE.
def make_synthetic(G, y, seed):
    rng = np.random.default_rng(seed)
    lam, U = np.linalg.eigh(L)
    # smooth ground truth = random combination of the 5 lowest-frequency eigenvectors
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam[:5]))
    # rescale so x has the same centre/spread as the real y
    x = np.median(y) + np.std(y) * x / np.std(x)
    sigma = 0.1 * np.std(y)
    y_syn = x + rng.normal(0, sigma, n)               # Gaussian noise
    idx = rng.choice(n, 3, replace=False)             # 3 outlier nodes
    y_syn[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, y_syn

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))

# (b) Parameter selection on REAL data: hide 1/5 of the nodes, predict them again.
def cv_score(solve, y, params, k=5, seed=0):
    folds = np.random.default_rng(seed).permutation(n) % k
    err = []
    for f in range(k):
        mask = folds != f                              # True = observed
        x = solve(G, y, mask=mask, **params)
        err.append(np.abs(x[~mask] - y[~mask]))
    return np.median(np.concatenate(err))              # median => robust to outliers in y

In [ ]:
# ---- Evaluation helpers (used in section 4 and the appendices) ---------------
def cv_mean(params, seeds=range(5)):
    # cv_score averaged over 5 different random splits, so one lucky/unlucky split does not decide. Lower = better.
    return np.mean([cv_score(solve_two_stage, y, params, seed=sd) for sd in seeds])

def synth_rmse(params, seeds=range(20)):
    # RMSE against the KNOWN x on synthetic data, one value per seed (seeds 0-19, as agreed in the group doc).
    out = []
    for sd in seeds:
        x_t, y_s = make_synthetic(G, y, sd)
        out.append(rmse(solve_two_stage(G, y_s, **params), x_t))
    return np.array(out)

## 2. The method

### Assumptions
1. **Smooth signal:** the true `x` changes slowly over the graph; close sensors have similar values, and real features are broader than one node.
2. **Observation model:** `y = x + ε + o`. `ε` = small Gaussian noise at every node; `o` = large error at a *few* nodes (outliers), rare and not clustered.
3. **Edge weights are similarities:** on the data `w = exp(−a·d²)` of the distance between sensors (checked on all 100 cases).
4. A flag means *suspected* outlier: a lone value that disagrees with all its neighbours is treated as an outlier, so a real peak narrower than the sensor spacing is erased.

### Stage 1: flag outliers
- `m_i` = weighted median of the neighbours' `y_j` (weights `w_ij`).
- Residual `r_i = y_i − m_i`; centre `c = median(r)`.
- Robust scale `s = 1.4826 · median(|r_i − c|)` (1.4826·MAD ≈ σ for Gaussian data).
- Flag node `i` if `|r_i − c| > k·s`, with `k = 2.5` (MAD rule, "moderately conservative", Leys et al. 2013).
- **Prune pass:** re-judge the flagged nodes one at a time: confirm the worst, recompute the medians without the confirmed nodes, repeat until no
  candidate exceeds `k·s` (`s` kept fixed); candidates never confirmed are un-flagged. Removes "victims" of neighbouring outliers, never adds flags.

### Stage 2: refill the flagged nodes
- Kept nodes `U`: `x_U = y_U`. Flagged nodes `F`: minimise the roughness `xᵀLx = Σ w_ij (x_i − x_j)²` over `x_F` (`L = D − W`).
- Solution: `L_FF x_F = −L_FU y_U`, i.e. every flagged node = weighted average of its neighbours, `x_i = Σ_j w_ij x_j / Σ_j w_ij` (harmonic interpolation).
- Optional (`lam > 0`, not used in the main result): `min Σ_{i∈U} (x_i − y_i)² + lam·xᵀLx` → `(M + lam·L) x = M y`.

### Stage 1 variants tested (Appendix C, `k = 2.5`, `lam = 0`)

| variant | synthetic RMSE (100 graphs × 5 seeds) | outliers caught | false flags / run | real cases with adjacent flags |
|---|---|---|---|---|
| none (`x_hat = y`) | 0.2956 | – | – | – |
| original | 0.2471 | 52 % | 1.26 | 10 |
| **prune (used)** | **0.2443** | 49 % | **1.05** | **7** |
| full iterative | 0.2759 | 51 % | 1.74 | 10 |
| floor on `s` | 0.2907 | 9 % | 0.19 | 8 |

In [ ]:
def weighted_median(values, weights):
    # Computes m_i from a node's neighbours. `values` = y_j of the neighbours, `weights` = w_ij.
    # Example (worked example above): values=[10,11,12,50], weights=[1,3,1,1]  ->  returns 11.
    order = np.argsort(values)                         # Step 3: sort neighbours by value (small -> large)
    v, w = np.asarray(values)[order], np.asarray(weights)[order]
    c = np.cumsum(w)                                   # Step 4: running total of weights, e.g. [1, 4, 5, 6]
    half = 0.5 * c[-1]                                 #         half of the total weight, e.g. 3
    return v[np.searchsorted(c, half)]                 #         first value whose running total reaches `half`

def robust_spread(v):
    # 1.4826 * MAD: a standard-deviation-like spread that outliers barely affect.
    v = np.asarray(v, dtype=float)
    return 1.4826 * np.median(np.abs(v - np.median(v)))

def detect_outliers(G, y, mask=None, k=2.5, s_min=0.0, s_fixed=None):
    # Stage 1 (one pass). Returns (flagged boolean array, residuals r, robust scale s).
    # `mask` (True = observed) lets cv_score hide nodes: hidden nodes are NOT used as neighbours
    # and are never flagged (they are "unknown", handled in stage 2).
    # `s_min`  : floor for s (improvement 2). `s_fixed`: use this s instead of estimating it (used by the iterative version).
    n = len(y)
    obs = np.ones(n, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)

    # m[i] = "what node i should read, judged only by its neighbours" (weighted median, see section 2)
    m = np.full(n, np.nan)
    for i in range(n):
        nb = [j for j in G.neighbors(i) if obs[j]]     # Step 1: node ids j of i's neighbours (observed only)
        if nb:                                         # a node whose neighbours are all hidden gets no m_i
            vals = y[nb]                               # Step 2: neighbours' observed values y_j ...
            wts = [G[i][j]["weight"] for j in nb]      #         ... and edge weights w_ij
            m[i] = weighted_median(vals, wts)          # Steps 3-4: sort + walk to half the total weight

    r = y - m                                          # residual vs neighbourhood
    valid = obs & ~np.isnan(r)                         # nodes we can actually judge
    r0 = np.median(r[valid])                           # centre of residuals (usually ~0)
    if s_fixed is None:
        s = robust_spread(r[valid])                    # robust sigma (1.4826 * MAD)
        s = max(s, s_min, 1e-8 * np.std(y))            # floor (improvement 2) + guard against s = 0
    else:
        s = s_fixed
    flagged = valid & (np.abs(r - r0) > k * s)
    return flagged, r, s

def detect_outliers_iterative(G, y, mask=None, k=2.5, s_min=0.0, only_first_pass=False):
    # Improvement 1: flag ONE node at a time (the worst), then recompute every neighbour median WITHOUT it.
    # A clean node that only looked odd because an outlier sat among its neighbours ("victim") is then judged again
    # with that outlier removed, and usually stops looking odd.
    # s is estimated once on the first pass and then kept fixed, so removing outliers does not tighten the threshold.
    n = len(y)
    obs = np.ones(n, dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    first, r, s = detect_outliers(G, y, mask=obs, k=k, s_min=s_min)  # first pass: s (and the first-pass flags)
    flagged = np.zeros(n, dtype=bool)
    for _ in range(n):
        trusted = obs & ~flagged                       # flagged nodes no longer count as neighbours
        cand, r, _ = detect_outliers(G, y, mask=trusted, k=k, s_fixed=s)
        if only_first_pass:
            cand &= first                              # option: only re-judge first-pass flags (can remove victims, never adds new flags)
        if not cand.any():
            break
        r0 = np.median(r[trusted & ~np.isnan(r)])
        dev = np.where(cand, np.abs(r - r0), -np.inf)
        flagged[np.argmax(dev)] = True                 # flag only the single worst node, then re-judge the rest
    return flagged, r, s

def inpaint(L, y, keep, lam=0.0):
    # Stage 2. `keep` = boolean array of nodes whose y we trust (the set U). Others (the set F) are re-estimated.
    n = len(y)
    if not keep.any():                                 # nothing trusted: fall back to a constant
        return np.full(n, np.median(y))
    F = ~keep
    if not F.any():
        return y.copy()
    if lam == 0.0:
        x = y.copy()                                   # Stage 2, Step 1: x_U = y_U (kept nodes copy their observation)
        L_FF = L[np.ix_(F, F)]                         # Stage 2, Step 4: Laplacian block flagged x flagged
        L_FU = L[np.ix_(F, keep)]                      #         Laplacian block flagged x kept
        x[F] = np.linalg.solve(L_FF, -L_FU @ y[keep])  #         solve L_FF x_F = -L_FU y_U (each x_i = weighted avg of neighbours)
        return x
    # Extension: minimise sum_{i in U} (x_i - y_i)^2 + lam * x^T L x  ->  (M + lam L) x = M y
    M = np.diag(keep.astype(float))                    # 1 for trusted nodes, 0 for flagged ones
    return np.linalg.solve(M + lam * L, M @ y)

def flag_nodes(G, y, mask=None, k=2.5, iterative=False, s_floor=0.0):
    # iterative: False | True ("full") | "prune" (only re-judge first-pass flags)
    # Stage 1 with the chosen options. s_floor is RELATIVE: s >= s_floor * robust spread of the observed y.
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    s_min = s_floor * robust_spread(y[obs])
    if iterative:
        return detect_outliers_iterative(G, y, mask=obs, k=k, s_min=s_min, only_first_pass=(iterative == "prune"))
    return detect_outliers(G, y, mask=obs, k=k, s_min=s_min)

def solve_two_stage(G, y, mask=None, k=2.5, lam=0.0, iterative="prune", s_floor=0.0):
    # Contract of the group doc: solve_<name>(G, y, mask=None, **params) -> ndarray of length n.
    # Defaults = the main result (k=2.5, lam=0, iterative="prune"). iterative=False is the original Stage 1; True and s_floor > 0 are the other
    # Stage 1 improvements tested in section 2 ("Improving Stage 1") and Appendix C.
    y = np.asarray(y, dtype=float)
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    flagged, _, _ = flag_nodes(G, y, mask=obs, k=k, iterative=iterative, s_floor=s_floor)
    keep = obs & ~flagged                              # trusted = observed AND not an outlier
    L_G = nx.laplacian_matrix(G, nodelist=range(len(y)), weight="weight").toarray()   # Laplacian of THIS graph
    return inpaint(L_G, y, keep, lam=lam)

### Functions for the group

| function | input | output |
|---|---|---|
| `flag_outliers(G, y, k=2.5, mask=None, prune=True, return_details=False)` | graph, observed `y` (length n), optional `mask` (True = observed) | bool array, True = suspected outlier (+ dict of `neighbour_median`, `residual`, `z`, `s` if `return_details=True`) |
| `refill_outliers(G, y, flagged, lam=0.0)` | graph, `y`, bool array of untrusted nodes | `x_hat` (length n): flagged nodes refilled, others unchanged (`lam = 0`) |

To use only the flags in another method, pass them as a mask: `solve_<yours>(G, y, mask=~flag_outliers(G, y))`, and re-run the flagging inside every `cv_score` fold.

In [ ]:
def flag_outliers(G, y, k=2.5, mask=None, prune=True, return_details=False):
    """Stage 1 of method 6: decide which nodes are suspected outliers.

    Inputs
    ------
    G : networkx.Graph
        Nodes 0 .. n-1. Every edge has a "weight" (bigger = closer, more similar sensors).
    y : array-like of length n
        Observed values; y[i] belongs to node i.
    k : float, default 2.5
        Threshold, in "typical gaps" s. Larger k = fewer flags.
    mask : bool array of length n, optional
        True = observed, False = hidden/missing. Hidden nodes are not used as neighbours and are never flagged.
        Default: all nodes observed.
    prune : bool, default True
        Re-judge the flagged nodes one at a time and un-flag "victims" (clean nodes that only looked odd
        because outliers sat among their neighbours).
    return_details : bool, default False
        Also return the numbers behind the decision.

    Output
    ------
    flagged : bool array of length n
        True = suspected outlier: do not trust y[i].
    details : dict (only if return_details=True), arrays of length n from the first pass (before pruning):
        "neighbour_median" (m_i), "residual" (r_i = y_i - m_i), "z" ((r_i - median(r)) / s), and "s" (a float).

    Example
    -------
    flagged = flag_outliers(G, y)
    np.flatnonzero(flagged)          # node ids of the suspected outliers
    """
    y = np.asarray(y, dtype=float)
    obs = np.ones(len(y), dtype=bool) if mask is None else np.asarray(mask, dtype=bool)
    y_safe = np.where(obs, y, np.nanmedian(y[obs]))    # values of hidden nodes are never used; avoids NaN issues
    flagged, _, _ = flag_nodes(G, y_safe, mask=obs, k=k, iterative="prune" if prune else False)
    if not return_details:
        return flagged
    _, r, s = detect_outliers(G, y_safe, mask=obs, k=k)
    valid = obs & ~np.isnan(r)
    z = (r - np.median(r[valid])) / s
    return flagged, {"neighbour_median": y_safe - r, "residual": r, "z": z, "s": s}

def refill_outliers(G, y, flagged, lam=0.0):
    """Stage 2 of method 6: compute new values for the flagged nodes.

    Inputs
    ------
    G : networkx.Graph
        Same graph as for flag_outliers.
    y : array-like of length n
        Observed values (entries of flagged nodes are ignored, they may be NaN).
    flagged : bool array of length n
        True = do not trust y[i]: the outliers from flag_outliers, and also any missing/hidden node.
    lam : float, default 0
        0  -> only flagged nodes change: each becomes the weighted average of its neighbours
              (harmonic interpolation: solve L_FF x_F = -L_FU y_U).
        >0 -> also smooth the trusted nodes: solve (M + lam*L) x = M y.

    Output
    ------
    x_hat : float array of length n
        With lam = 0: x_hat[i] = y[i] for trusted nodes, and the new value for flagged nodes.

    Example
    -------
    flagged = flag_outliers(G, y)
    x_hat = refill_outliers(G, y, flagged)
    {int(i): x_hat[i] for i in np.flatnonzero(flagged)}   # new values of the outliers
    """
    y = np.asarray(y, dtype=float)
    flagged = np.asarray(flagged, dtype=bool)
    y_safe = np.where(flagged, 0.0, y)                 # untrusted values never enter the result
    L_G = nx.laplacian_matrix(G, nodelist=range(len(y)), weight="weight").toarray()
    return inpaint(L_G, y_safe, ~flagged, lam=lam)

In [ ]:
# Example: flag, inspect the numbers, refill. Then check it matches solve_two_stage exactly.
flagged, info = flag_outliers(G, y, return_details=True)
x_new = refill_outliers(G, y, flagged)
print("suspected outliers:", np.flatnonzero(flagged))
for i in np.flatnonzero(flagged):
    print(f"  node {i}: y = {y[i]:.3f}, neighbours' median = {info['neighbour_median'][i]:.3f}, "
          f"z = {info['z'][i]:+.2f}  ->  new value {x_new[i]:.3f}")
assert np.allclose(x_new, solve_two_stage(G, y, k=2.5, lam=0.0, iterative="prune"))
print("same result as solve_two_stage: OK")

## 3. Final result on the real data

Parameters: `k = 2.5`, `lam = 0`, `iterative = "prune"`. Set the group's real MSSV in section 1 first.

In [ ]:
FINAL_PARAMS = {"k": 2.5, "lam": 0.0, "iterative": "prune"}   # main result of method 6 (justification in the cell above)

x_hat_real = solve_two_stage(G, y, **FINAL_PARAMS)
flagged_real, r_real, s_real = flag_nodes(G, y, k=FINAL_PARAMS["k"], iterative=FINAL_PARAMS["iterative"])
assert x_hat_real.shape == y.shape and np.all(np.isfinite(x_hat_real))

np.save(RES_DIR / "xhat_6.npy", x_hat_real)
print("flagged (suspected outlier) nodes on real data:", np.flatnonzero(flagged_real))
for i in np.flatnonzero(flagged_real):
    print(f"  node {i}: y = {y[i]:.3f} -> x_hat = {x_hat_real[i]:.3f}")

In [ ]:
# Figure: y vs x_hat on the graph; flagged nodes circled in red. Same colour scale for both panels.
vmin, vmax = min(y.min(), x_hat_real.min()), max(y.max(), x_hat_real.max())
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, vals, title in [(axes[0], y, "Observed y"), (axes[1], x_hat_real, "Recovered x_hat (method 6)")]:
    nodes = nx.draw_networkx_nodes(G, positions, node_color=vals, cmap="viridis",
                                   vmin=vmin, vmax=vmax, node_size=380, ax=ax)
    nx.draw_networkx_edges(G, positions, alpha=0.3, ax=ax)
    nx.draw_networkx_labels(G, positions, font_size=8, ax=ax)
    ax.set_title(title); ax.axis("off")
nx.draw_networkx_nodes(G, positions, nodelist=list(np.flatnonzero(flagged_real)), node_color="none",
                       edgecolors="red", linewidths=2.5, node_size=520, ax=axes[0])
fig.colorbar(nodes, ax=axes, label="value", shrink=0.8)
plt.savefig(FIG_DIR / "fig_y_vs_xhat_method6.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Result row for the group comparison table

One row per method, as agreed in the group doc: method, parameters, synthetic RMSE (mean ± std over seeds 0-19), `cv_score` on the real data.
(How these two numbers are computed and how to read them: Appendix A and B.)

In [ ]:
synth_final = synth_rmse(FINAL_PARAMS)
result_row = {
    "method": "6. Two-stage: median/MAD outlier filter + harmonic inpainting",
    "params": FINAL_PARAMS,
    "synthetic_RMSE_mean": round(float(synth_final.mean()), 4),
    "synthetic_RMSE_std":  round(float(synth_final.std()), 4),
    "cv_score": round(float(cv_mean(FINAL_PARAMS)), 4),
}
print(result_row)

## 5. Write-up (Mục VI skeleton — edit in your own words, 1-1.5 pages)

**1. Assumptions.** The true signal `x` is smooth on the graph (neighbouring nodes have similar values). Observation = `x` + small Gaussian noise on all nodes + a *few* large outliers. Outliers are rare and not clustered, so most neighbours of any node are clean. Edge weights are a Gaussian kernel of the distance between sensors (heavier edge = closer, more similar sensors).

**2. Procedure / loss.**
Stage 1: `m_i` = weighted median of neighbours' `y`; `r_i = y_i - m_i`; `s = 1.4826·MAD(r)`; flag if `|r_i - med(r)| > k·s`; then re-judge the flagged nodes one at a time with the others removed from the medians, and un-flag those that no longer exceed the threshold ("victims" of neighbouring outliers).
Stage 2: with `F` flagged and `U` kept, `x_U = y_U` and `x_F = argmin x^T L x`, i.e. `L_FF x_F = -L_FU y_U`.
(Optional: `min Σ_U (x_i - y_i)^2 + lam x^T L x`.)

**3. Optimisation and parameters.** Closed form (one linear solve with `L_FF`, which is positive definite because the graph is connected and `U` is non-empty). `k = 2.5` from the robust-statistics literature (false positive rate ≈ 1.2 %), checked with 5-fold `cv_score` (Appendix A). Sanity check: flagged nodes on synthetic data should match the planted outliers (Appendix B).

**4. Experiments and limits.** Report the result row (section 4), the comparison tables of Appendix A and B, and the two figures. Limits: (a) the detector assumes most neighbours are clean, so adjacent outliers can hide each other; (b) with `lam=0` kept nodes keep their Gaussian noise, so the method cannot beat the noise floor on them; (c) a too-small `k` flags good nodes (loses information), a too-large `k` misses outliers; (d) with only a few nodes, MAD is itself noisy; (e) on 32 nodes the neighbour median uses few neighbours (about 5).

---
# Appendices (optional reading)

The main result above does not need them to be read. They contain the evidence behind the parameter choice:
- **Appendix A**: what `cv_score` is, and the grid search over `k` and `lam` on the real data.
- **Appendix B**: synthetic data with a known `x`: a sanity check on one seed and RMSE over seeds 0-19.
- **Appendix C**: the Stage 1 variants (original, prune, full iterative, floor on `s`) compared on all 100 public graphs.

Run them in order (B uses the variant found in A).

## Appendix A. Parameter check with `cv_score` (real data)

`cv_score` (shared code): 5 folds; hide 1/5 of the nodes, run the whole method on the rest (flagging included), error `|x_hat − y|` on the hidden nodes;
median over all nodes; `cv_mean` averages 5 random splits. Lower is better. It compares with noisy `y` (noise floor) and is noisy on 32 nodes.
Grid over `k` and `lam`; `k = 1e9` = no outlier filter.

In [ ]:
k_grid   = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 1e9]   # 1e9 = never flag => "no filter" baseline
lam_grid = [0.0, 0.05, 0.2, 1.0]

it = FINAL_PARAMS["iterative"]                              # same Stage 1 variant as the main result
table = {(k, lam): cv_mean({"k": k, "lam": lam, "iterative": it}) for k in k_grid for lam in lam_grid}
print("cv_score (rows k, cols lam)  lam:", lam_grid)
for k in k_grid:
    print(f"k={k:<7}", [round(float(table[(k, lam)]), 4) for lam in lam_grid])

best_pure = min(((k, 0.0) for k in k_grid), key=lambda p: table[p])
best_any  = min(table, key=table.get)
params_variant = {"k": best_any[0], "lam": best_any[1], "iterative": it}   # best smoothing variant, compared in Appendix B
print("chosen (main)     :", (FINAL_PARAMS["k"], FINAL_PARAMS["lam"]), "->", round(float(table[(FINAL_PARAMS["k"], FINAL_PARAMS["lam"])]), 4))
print("best pure (lam=0) :", best_pure, "->", round(float(table[best_pure]), 4))
print("best overall      :", best_any,  "->", round(float(table[best_any]), 4))

In [ ]:
# cv_score curve vs k (pure method, lam = 0)
ks = [k for k in k_grid if k < 1e8]
plt.figure(figsize=(6, 3.5))
plt.plot(ks, [table[(k, 0.0)] for k in ks], "o-", label="two-stage, lam=0")
plt.axhline(table[(1e9, 0.0)], color="gray", ls="--", label="no outlier filter")
plt.axvline(FINAL_PARAMS["k"], color="red", ls=":", label=f"chosen k = {FINAL_PARAMS['k']}")
plt.xlabel("threshold k"); plt.ylabel("cv_score (median |error|)"); plt.legend(); plt.grid(alpha=.3)
plt.title("Checking k by cross-validation")
plt.savefig(FIG_DIR / "fig_cv_k_method6.png", dpi=150, bbox_inches="tight"); plt.show()

## Appendix B. Synthetic data (known `x`)

`make_synthetic` (shared code, section 1) builds a smooth `x` on our graph, adds Gaussian noise and 3 outliers, so RMSE against the true `x` can be measured.
This only reflects the generator's assumptions, not necessarily the real data.

### B.1 Sanity check on one seed
Are the planted outliers (largest `|y_syn − x_true|`) among the flagged nodes?

In [ ]:
x_true, y_syn = make_synthetic(G, y, seed=0)
flagged, r, s = flag_nodes(G, y_syn, k=FINAL_PARAMS["k"], iterative=FINAL_PARAMS["iterative"])
x_hat = solve_two_stage(G, y_syn, **FINAL_PARAMS)
print("flagged nodes:", np.flatnonzero(flagged))
print("planted outliers (largest |y_syn - x_true|):", np.argsort(-np.abs(y_syn - x_true))[:3])
print("RMSE of y itself  :", round(rmse(y_syn, x_true), 4))
print("RMSE of two-stage :", round(rmse(x_hat, x_true), 4))

### B.2 RMSE over seeds 0-19
Baseline (`x_hat = y`, no processing), the main parameters, and the best variant from Appendix A.

In [ ]:
base = np.array([rmse(make_synthetic(G, y, sd)[1], make_synthetic(G, y, sd)[0]) for sd in range(20)])
r_main, r_var = synth_rmse(FINAL_PARAMS), synth_rmse(params_variant)

print(f"baseline (x_hat = y)              : {base.mean():.4f} ± {base.std():.4f}")
print(f"main    {FINAL_PARAMS}: {r_main.mean():.4f} ± {r_main.std():.4f}")
print(f"variant {params_variant}: {r_var.mean():.4f} ± {r_var.std():.4f}")

## Appendix C. Stage 1 variants on all 100 public graphs

Reproduces the table in section 2 (Stage 1, "Improving Stage 1"). Every graph in `public_cases.json` is used:
- **synthetic**: 5 seeds per graph (500 runs), generator identical to `make_synthetic` but on each graph; we know which 3 nodes are planted outliers;
- **real data**: flags on each case's real `y`, and `cv_score` averaged over 5 splits.

Takes about half a minute.

In [ ]:
with open(ROOT / "public_cases.json", encoding="utf-8") as f:
    ALL = json.load(f)["cases"]

def load_case(cid):
    c = ALL[cid]; Gc = nx.Graph()
    for i, xy in enumerate(c["positions"]): Gc.add_node(i, pos=tuple(map(float, xy)))
    for u, v, w in c["edges"]: Gc.add_edge(int(u), int(v), weight=float(w))
    return Gc, np.asarray(c["y"], dtype=float)

def synthetic_on(Gc, yc, seed):
    # Same recipe as make_synthetic (shared code), but for any graph; also returns the planted outlier nodes.
    nc = len(yc); Lc = nx.laplacian_matrix(Gc, nodelist=range(nc), weight="weight").toarray()
    rng = np.random.default_rng(seed); lam_, U = np.linalg.eigh(Lc)
    x = U[:, :5] @ (rng.normal(size=5) / (1 + lam_[:5])); x = np.median(yc) + np.std(yc) * x / np.std(x)
    sigma = 0.1 * np.std(yc); ys = x + rng.normal(0, sigma, nc)
    idx = rng.choice(nc, 3, replace=False); ys[idx] += rng.choice([-1, 1], 3) * rng.uniform(4, 8, 3) * sigma
    return x, ys, set(idx.tolist())

def cv_on(Gc, yc, params, seeds=range(5), k=5):
    out = []
    for sd in seeds:
        folds = np.random.default_rng(sd).permutation(len(yc)) % k; err = []
        for f in range(k):
            mk = folds != f; xh = solve_two_stage(Gc, yc, mask=mk, **params); err.append(np.abs(xh[~mk] - yc[~mk]))
        out.append(np.median(np.concatenate(err)))
    return np.mean(out)

VARIANTS = {"original": {"iterative": False}, "prune": {"iterative": "prune"}, "full iterative": {"iterative": True},
            "floor on s": {"iterative": False, "s_floor": 0.49}, "full iterative + floor": {"iterative": True, "s_floor": 0.49}}
case_ids = sorted(ALL, key=lambda x: (0, int(x)) if x.isdecimal() else (1, x))
graphs = {cid: load_case(cid) for cid in case_ids}

print(f"{'variant':24s} {'synth RMSE':>10s} {'caught':>7s} {'false/run':>9s} {'adj-flag cases':>14s} {'mean cv':>8s}")
for name, opt in VARIANTS.items():
    p = {"k": 2.5, "lam": 0.0, **opt}
    rm, hit, false, adj, cvs = [], 0, 0, 0, []
    for cid, (Gc, yc) in graphs.items():
        for sd in range(5):
            x, ys, planted = synthetic_on(Gc, yc, sd)
            F = set(np.flatnonzero(flag_nodes(Gc, ys, k=2.5, iterative=p.get("iterative", False), s_floor=p.get("s_floor", 0.0))[0]).tolist())
            rm.append(rmse(solve_two_stage(Gc, ys, **p), x)); hit += len(F & planted); false += len(F - planted)
        Fr = np.flatnonzero(flag_nodes(Gc, yc, k=2.5, iterative=p.get("iterative", False), s_floor=p.get("s_floor", 0.0))[0]).tolist()
        adj += any(len(cc) > 1 for cc in nx.connected_components(Gc.subgraph(Fr)))
        cvs.append(cv_on(Gc, yc, p))
    runs = 5 * len(graphs)
    print(f"{name:24s} {np.mean(rm):10.4f} {100*hit/(3*runs):6.0f}% {false/runs:9.2f} {adj:14d} {np.mean(cvs):8.4f}")